<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-01-assistant-configuration/notebook.ipynb)


# Session 1 — Configure the assistant and the repository instructions

**Goal:** turn a generic assistant into a project-aware collaborator, and practice the inspect → plan → edit → test → review loop.

This session runs in your coding assistant, not in this notebook. The notebook is your checklist and logbook.

In [5]:
# manual-run: edits assistant configuration — run with your assistant open
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = ollama (gemma3 at http://localhost:11434/v1)
ready. LIVE is the ollama lane.


In [7]:
from bootcamp_agent.checks import check, review

## 1. Warm-up: weak prompt vs project-aware prompt

In your assistant, ask both and paste short excerpts of the answers here:

- Weak: *"add a search feature"*
- Project-aware: *"read AGENTS.md, then propose a plan to add a tags filter to search_documents in src/bootcamp_agent/tools.py — plan only, no edits"*

**Paste both responses in the cell below and note the differences.**

weak : I’ll first locate the user-facing entry point and any existing query or retrieval flow, then I’ll wire search into the smallest relevant surface and run its focused check.The repository has several executable surfaces, but the likely product surface is the capstone agent/CLI rather than a web frontend. I’m checking those entry points and the existing retrieval module now; the key question is whether “search” should be a CLI command or a reusable corpus search API.

In [ ]:
project Aware: I’ll read the repository policy and the current `search_documents` implementation, then give you a focused implementation plan only. I’ll keep the scope to the tool boundary, its retrieval behavior, and the checks that would validate the change.

Read 

Searched for text `search_documents` (`**/src/bootcamp_agent/**`), 5 results

Read 

The policy confirms this should stay narrow, validate inputs at the tool boundary, preserve transparent retrieval context, and avoid adding dependencies or tests in the unpublished `tests/` directory. The current tool delegates ranking to `retrieve`, so I’m checking the document shape and retrieval API next to decide whether tag filtering belongs in the tool or the retrieval layer.

Searched for text `tags=` (`**/src/**`), 2 results

Searched for text `build_tools(` (`**/*.{py,md,ipynb}`), 2 results

Read 

Read 

Searched for regex `max_results|Search the corpus|No matching passages` (`**/*.{py,md}`), 7 results

Read 

The document model already exposes normalized `Document.tags`, while `retrieve` accepts a document sequence and computes ranking statistics over that sequence. The smallest coherent design is therefore to add an optional tag filter to `search_documents`, validate it at the boundary, filter the candidate documents before retrieval, and leave the retriever unchanged; the plan should explicitly settle whether multiple tags mean “any” or “all.”

**Plan**

1. Update `search_documents` in `tools.py` to accept an optional `tags` parameter while preserving existing callers:
   ```python
   tags: Sequence[str] | None = None
   ```

2. Validate and normalize tag input at the tool boundary:
   - Ignore surrounding whitespace.
   - Reject empty tag values.
   - Decide and document whether multiple tags use **all-match** or **any-match** semantics. I recommend **all-match**, which makes the filter useful for narrowing results.
   - Keep tag matching case-insensitive.

3. Filter the `Document` sequence before calling `retrieve`, so ranking and IDF are calculated only across documents matching the requested tags.

4. Preserve existing behavior and safeguards:
   - Empty queries still raise `ToolError`.
   - `max_results` remains clamped to `MAX_SEARCH_RESULTS`.
   - No-match results continue returning `"No matching passages found."`
   - No changes to `retrieval.py` or provider-specific code.

5. Update the tool description to mention tag filtering so the model can discover the capability.

6. Validate with:
   - `uv run ruff check src/bootcamp_agent/tools.py`
   - The repository’s targeted bootcamp check if this tool is covered by the relevant session.
   - Manual checks using tagged and untagged documents for matching, non-matching, multiple-tag, invalid-tag, and backward-compatible calls.

No files will be edited until you approve the plan.

## 2. The task loop, enforced by you

Feature: **add an optional `tags` filter to `search_documents`** — work in a scratch branch/copy.

- [ ] Ask for a **plan** first. Read it. Restrict files it may touch.
- [ ] Ask for the **smallest implementation**.
- [ ] Inspect the **diff** yourself, line by line.
- [ ] Verify with what this repository gives you: `uv run ruff check src/bootcamp_agent/tools.py`,
      then the `check(...)` cell below. (There is no `pytest` here — the suite holds the solved
      value of every exercise and is never published. `AGENTS.md` says so too.)
- [ ] **Reject at least one change** — unsafe, unnecessary, or out of scope — and record what you rejected and why below.
- [ ] Ask for a summary of remaining risks.

**Rejected change + reason:** *(fill in)*

## 3. Improve the instructions

 the assistant failed to address other invalid inputs, and I updated AGENTS.md to enforce comprehensive validation.

## 4. The loop, applied to this course

From tomorrow on, every exercise ends with a `check(...)` cell. Give your assistant the exercise's **Context** and **Instructions**, let it fill the `TODO(you)` lines, then run the check yourself. You read the verdict, not the assistant.

## 5. Exercise: the task loop, evidenced

**Context.** The loop only works if you can show you ran it. Four pieces of evidence, and the third one is the whole session: a change you refused. If you rejected nothing, you were not reviewing.

**Instructions.**

1. Work the feature (a `tags` filter on `search_documents`) in a scratch copy, through plan, edit, test, review.
2. Fill each field from what actually happened. `plan_approved` is filled as an example; replace it with yours.
3. Run the check. It refuses a blank rejection, because a loop with no rejection is not the loop.

In [2]:
loop ={
  "plan_approved": "It proposed editing only tools.py: add optional case-insensitive all-match tags filtering, preserve the result cap, and validate queries.",
  "diff_inspected": "Read AGENTS.md, tools.py, documents.py, retrieval.py, and pyproject.toml; inspected the tool boundary, Document.tags, retrieve(), and validation commands.",
  "rejected_change": "Changing retrieval.py or adding a new dependency.",
  "why_rejected": "The existing retrieval API already accepts filtered documents, so broader changes were unnecessary and out of scope.",
  "risks": "Ruff was unavailable in the active environment, and the repository's unpublished tests were not accessible; focused smoke tests, compilation, and diagnostics passed."
}
for key, value in loop.items():
    print(f"{key:18} {'(empty)' if not value else value[:58]}")

plan_approved      It proposed editing only tools.py: add optional case-insen
diff_inspected     Read AGENTS.md, tools.py, documents.py, retrieval.py, and 
rejected_change    Changing retrieval.py or adding a new dependency.
why_rejected       The existing retrieval API already accepts filtered docume
risks              Ruff was unavailable in the active environment, and the re


**Expected output** (yours may differ in wording, not in shape):

```
plan_approved      It proposed editing only tools.py: add an optional tags
diff_inspected     Six lines in search_documents plus one new test. I read
...
✅ ch01-e1 passed
```

In [9]:
check("ch01-e1", loop)

✅ ch01-e1 passed


True

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [10]:
review("ch01")

ch01: 1/1 passed  ·  100/100 marks


True

## Exit ticket

Homework: keep the improved instruction file; bring the rejected-change story to tomorrow's warm-up.